# Базовая проверка русского NER spaCy на russian-pii-66k

Тестовые записи зафиксированы в `russian_pii_66k_test_manifest.json`. Ноутбук проверяет SHA-256 исходного JSONL перед оценкой. Выборка не используется для обучения.

Русская модель spaCy различает `PER`, `LOC`, `ORG`. Здесь `GIVENNAME` и `SURNAME` сопоставлены с `PER`, а `CITY` и `STREET` — с `LOC`. Соседние части имени объединяются. Для этих групп считаются строгое совпадение границ и пересечение с однозначным сопоставлением сущностей. Остальные исходные типы перечисляются отдельно: отсутствие соответствующей метки у модели не является измерением качества обучаемого распознавания этих типов.

Разметка синтетического корпуса может содержать шум; результаты этого теста нельзя считать оценкой на реальных обращениях.

In [1]:
import hashlib
import json
from collections import Counter
from pathlib import Path

import spacy

# Ищем корень проекта при запуске из корня или из каталога ноутбука.
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'requirements.txt').is_file() and (p / 'notebooks/testing/russian_pii_66k_test_manifest.json').is_file()), None)
if root is None:
    raise FileNotFoundError('Не найден корень проекта')
manifest = json.loads((root / 'notebooks/testing/russian_pii_66k_test_manifest.json').read_text(encoding='utf-8'))
data_path = root / 'data/russian_pii_66k.jsonl'
if not data_path.is_file():
    raise FileNotFoundError(f'Нет {data_path}; запустите загрузчик с указанной в манифесте ревизией')
digest = hashlib.sha256(data_path.read_bytes()).hexdigest()
if digest != manifest['sha256']:
    raise ValueError('SHA-256 датасета не совпадает с манифестом; используйте закреплённую ревизию и версию datasets')
indices = manifest['indices']
if len(indices) != manifest['sample_size'] or indices != sorted(set(indices)):
    raise ValueError('В манифесте неверный список индексов')
selected = {}
selected_indices = set(indices)
with data_path.open(encoding='utf-8') as stream:
    for index, line in enumerate(stream):
        if index in selected_indices:
            selected[index] = json.loads(line)
if len(selected) != len(indices) or index + 1 != manifest['row_count']:
    raise ValueError('Размер датасета или тестовой выборки не совпадает с манифестом')
rows = [selected[index] for index in indices]
print(f"Ревизия: {manifest['revision']} | тестовых записей: {len(rows)} | SHA-256: {digest}")

Ревизия: d458b5a2e299d2e16adbd3b3921487f652b06a96 | тестовых записей: 1000 | SHA-256: 0b02943c2c090f448ed48549613ca412987d7ea4b35de2ff66dbaab2aecfe692


In [2]:
model_name = 'ru_core_news_sm'
try:
    nlp = spacy.load(model_name)
except OSError as error:
    raise RuntimeError('Установите модель: venv/bin/python -m spacy download ru_core_news_sm') from error
if nlp.meta['version'] != '3.8.0':
    raise ValueError('Для сравнимого запуска требуется ru_core_news_sm 3.8.0')
model_labels = set(nlp.get_pipe('ner').labels)
print(f"spaCy {spacy.__version__}; модель {model_name} {nlp.meta['version']}; NER: {sorted(model_labels)}")
docs = list(nlp.pipe((row['source_text'] for row in rows), batch_size=64))

spaCy 3.8.16; модель ru_core_news_sm 3.8.0; NER: ['LOC', 'ORG', 'PER']


In [3]:
LABEL_MAP = {'GIVENNAME': 'PER', 'SURNAME': 'PER', 'CITY': 'LOC', 'STREET': 'LOC'}


def gold_spans(row):
    """Сопоставляет поддерживаемые метки и объединяет соседние части имени."""
    text = row['source_text']
    spans = sorted((item['start'], item['end'], LABEL_MAP[item['label']]) for item in row['privacy_mask'] if item['label'] in LABEL_MAP)
    merged = []
    for start, end, label in spans:
        if label == 'PER' and merged and merged[-1][2] == 'PER' and 0 < start - merged[-1][1] <= 3 and text[merged[-1][1]:start].isspace():
            merged[-1] = (merged[-1][0], end, label)
        else:
            merged.append((start, end, label))
    return set(merged)


def prf(true_positives, predicted, expected):
    """Вычисляет точность, полноту и F1 для точных совпадений границ."""
    precision = true_positives / predicted if predicted else 0.0
    recall = true_positives / expected if expected else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def overlap_matches(gold, predicted):
    """Сопоставляет пересекающиеся сущности одного типа по одной."""
    candidates = sorted((-(min(g[1], p[1]) - max(g[0], p[0])), g, p) for g in gold for p in predicted if g[2] == p[2] and min(g[1], p[1]) > max(g[0], p[0]))
    used_gold, used_pred = set(), set()
    for _, g, p in candidates:
        if g not in used_gold and p not in used_pred:
            used_gold.add(g)
            used_pred.add(p)
    return len(used_gold)


scores = {label: Counter() for label in ('PER', 'LOC')}
overlap_hits = Counter()
source_counts = Counter()
for row, doc in zip(rows, docs, strict=True):
    source_counts.update(item['label'] for item in row['privacy_mask'])
    gold = gold_spans(row)
    predicted = {(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents if ent.label_ in scores}
    for label in scores:
        scores[label]['gold'] += sum(span[2] == label for span in gold)
        scores[label]['pred'] += sum(span[2] == label for span in predicted)
        scores[label]['tp'] += sum(span[2] == label for span in gold & predicted)
        overlap_hits[label] += overlap_matches({span for span in gold if span[2] == label}, {span for span in predicted if span[2] == label})

print('Группа | Gold | Pred | TP | Precision | Recall | F1')
for label, counts in scores.items():
    p, r, f1 = prf(counts['tp'], counts['pred'], counts['gold'])
    print(f"{label:6} | {counts['gold']:4} | {counts['pred']:4} | {counts['tp']:3} | {p:.3f} | {r:.3f} | {f1:.3f}")
total = sum(scores.values(), Counter())
p, r, f1 = prf(total['tp'], total['pred'], total['gold'])
print(f"Итого  | {total['gold']:4} | {total['pred']:4} | {total['tp']:3} | {p:.3f} | {r:.3f} | {f1:.3f}")
print('\nПри пересечении границ (одно предсказание на одну сущность):')
for label, counts in scores.items():
    p, r, f1 = prf(overlap_hits[label], counts['pred'], counts['gold'])
    print(f'{label}: precision={p:.3f}, recall={r:.3f}, F1={f1:.3f}')
print('\nТипы датасета без сопоставления с метками этой модели:')
for label, count in sorted(source_counts.items()):
    if label not in LABEL_MAP:
        print(f'{label}: {count}')

Группа | Gold | Pred | TP | Precision | Recall | F1
PER    |  445 |  519 | 424 | 0.817 | 0.953 | 0.880
LOC    |  364 |  246 |   0 | 0.000 | 0.000 | 0.000
Итого  |  809 |  765 | 424 | 0.554 | 0.524 | 0.539

При пересечении границ (одно предсказание на одну сущность):
PER: precision=0.836, recall=0.975, F1=0.900
LOC: precision=0.923, recall=0.624, F1=0.744

Типы датасета без сопоставления с метками этой модели:
ACCOUNTNUM: 202
BUILDINGNUM: 140
CREDITCARDNUMBER: 119
DATEOFBIRTH: 149
DRIVERLICENSENUM: 92
EMAIL: 271
IDCARDNUM: 132
PASSWORD: 137
SOCIALNUM: 117
TAXNUM: 158
TELEPHONENUM: 253
USERNAME: 220
ZIPCODE: 95
